# SpikeGate — MaxFormer 10-384-T4 Analysis (A100)

Full SpikeGate framework pipeline on the **MaxFormer 10-384-T4** checkpoint.

| Phase | Description | Images |
|---|---|---|
| Calibration | AutoProfiler — Q/K/V sparsity, dead/ghost neurons, STBP timing | 500 |
| Stability | Multi-run gating fidelity & compute savings | 10 × 500 |
| Ablation | Per-policy fidelity/savings across all runs | 10 × 500 |
| Importance | MASK-ONE-OUT per head, 60 heads total | 10 × 500 |
| Report | 8 publication-quality figures + Markdown report | — |

**Runtime**: A100 GPU (40 GB or 80 GB)  
**Estimated wall time**: ~25–40 minutes on A100

## Step 1 — Verify A100 GPU

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Runtime → Change runtime type → A100 GPU")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU      : {gpu_name}")
print(f"VRAM     : {vram_gb:.1f} GB")

if 'A100' not in gpu_name:
    print(f"\nWARNING: Got '{gpu_name}', not A100. Analysis will still run but"
          " batch size may need reducing if OOM.")
else:
    print("\nA100 confirmed — optimal settings will be applied.")

## Step 2 — Clone Repository & Install Dependencies

In [ ]:
import os

REPO     = "SNN-Transformer-Knowlege-Extraction"
BASE_DIR = f"/content/{REPO}/MaxFormer"

if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/SiddheshUttarwar/{REPO}.git
else:
    print(f"{REPO} already cloned — skipping")

# os.chdir works correctly; %cd does NOT expand Python variables like {REPO}
os.chdir(BASE_DIR)
print(f"Working dir: {os.getcwd()}")

# Install all required packages
!pip install -q \
    "spikingjelly==0.0.0.0.12" \
    "timm==0.6.12" \
    "datasets" \
    "torchvision" \
    "seaborn" \
    "einops==0.7.0" \
    "tqdm"

print("\nInstallation complete.")

## Step 3 — Mount Google Drive & Copy Checkpoint

Upload `10-384-T4.pth.tar` to your Google Drive root folder (`My Drive/`) before running this cell.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# --- UPDATE THIS if your checkpoint is in a sub-folder ---
CKPT_ON_DRIVE = '/content/drive/MyDrive/10-384-T4.pth.tar'
CKPT_LOCAL    = './checkpoints/10-384-T4.pth.tar'
# ----------------------------------------------------------

os.makedirs('checkpoints', exist_ok=True)

if os.path.exists(CKPT_LOCAL):
    print(f"Checkpoint already present at {CKPT_LOCAL}")
elif os.path.exists(CKPT_ON_DRIVE):
    shutil.copy(CKPT_ON_DRIVE, CKPT_LOCAL)
    print(f"Checkpoint copied: {CKPT_ON_DRIVE} → {CKPT_LOCAL}")
else:
    raise FileNotFoundError(
        f"Checkpoint not found at '{CKPT_ON_DRIVE}'.\n"
        "Please update CKPT_ON_DRIVE or drag-and-drop the file into checkpoints/"
    )

## Step 4 — A100 Configuration & Python Path Setup

In [ ]:
import sys, os, warnings, torch

# ── A100 backend optimisations ───────────────────────────────────────
torch.backends.cuda.matmul.allow_tf32  = True
torch.backends.cudnn.allow_tf32        = True
torch.backends.cudnn.benchmark         = True
# ─────────────────────────────────────────────────────────────────────

# ── Analysis configuration ───────────────────────────────────────────
NUM_RUNS       = 10
IMAGES_PER_RUN = 500
A100_BATCH     = 32
REPORT_DIR     = 'analysis_outputs/comprehensive_report'
NUM_BLOCKS     = 10
T_STEPS        = 4
# Model constants (defined here so data cell can use them before model loads)
IMG_SIZE    = 224
NUM_HEADS   = 6
NUM_CLASSES = 1000
EMBED_DIMS  = 384
# ─────────────────────────────────────────────────────────────────────

# ── Ensure correct working directory & Python paths ──────────────────
BASE_DIR = "/content/SNN-Transformer-Knowlege-Extraction/MaxFormer"
os.chdir(BASE_DIR)

for p in [BASE_DIR, os.path.join(BASE_DIR, 'imagenet')]:
    if p not in sys.path:
        sys.path.insert(0, p)
# ─────────────────────────────────────────────────────────────────────

warnings.filterwarnings('ignore')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Working dir : {os.getcwd()}")
print(f"sys.path[0] : {sys.path[0]}")
print(f"sys.path[1] : {sys.path[1]}")
print(f"Device      : {device.upper()}")
if device == 'cuda':
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Batch size  : {A100_BATCH}  (was 10 in local config)")
print(f"Runs × imgs : {NUM_RUNS} × {IMAGES_PER_RUN} = {NUM_RUNS * IMAGES_PER_RUN} total")
print("TF32        : enabled (A100 matmul + cuDNN)")

## Step 5 — Load MaxFormer 10-384-T4

In [ ]:
import torch
import spikingjelly.clock_driven.neuron as _sj_neuron
from max_former import Max_Former

CHECKPOINT = './checkpoints/10-384-T4.pth.tar'

# Force torch backend for SpikingjJelly (Colab GPU compatible)
_orig_init = _sj_neuron.MultiStepLIFNode.__init__
def _patched_init(self, *args, **kwargs):
    kwargs['backend'] = 'torch'
    _orig_init(self, *args, **kwargs)
_sj_neuron.MultiStepLIFNode.__init__ = _patched_init

print(f"Loading checkpoint: {CHECKPOINT}")
ckpt       = torch.load(CHECKPOINT, map_location='cpu')
state_dict = ckpt.get('state_dict', ckpt)

model = Max_Former(
    in_channels=3, num_classes=NUM_CLASSES,
    embed_dims=EMBED_DIMS, mlp_ratios=4,
    depths=NUM_BLOCKS, T=T_STEPS
)
cleaned = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(cleaned, strict=False)
model.to(device).eval()

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded on : {device.upper()}")
print(f"Parameters      : {total_params:.1f} M")
print(f"Architecture    : MaxFormer {NUM_BLOCKS}-{EMBED_DIMS}-T{T_STEPS}")
if device == 'cuda':
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"VRAM used       : {allocated:.2f} GB")

## Step 6 — Stream ImageNet-1K Validation Data

In [ ]:
import torch
from torchvision import transforms
from datasets import load_dataset

def create_data_chunks(num_runs: int, images_per_run: int, batch_size: int):
    """Stream ImageNet-1K val and yield one list-of-batches per run."""
    dataset = load_dataset(
        'mrm8488/ImageNet1K-val', split='train',
        streaming=True,
    )
    tfs = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])
    batches_per_run = images_per_run // batch_size
    buf, run_batches = [], []
    for item in dataset:
        img = item['image']
        if img.mode != 'RGB':
            img = img.convert('RGB')
        buf.append(tfs(img).unsqueeze(0))
        if len(buf) == batch_size:
            run_batches.append((torch.cat(buf, dim=0), torch.zeros(batch_size)))
            buf = []
            if len(run_batches) == batches_per_run:
                yield run_batches
                run_batches = []
                num_runs -= 1
                if num_runs <= 0:
                    return

total_images = NUM_RUNS * IMAGES_PER_RUN
print(f"Streaming {NUM_RUNS} × {IMAGES_PER_RUN} = {total_images} images "
      f"(batch_size={A100_BATCH})...")

chunks = list(create_data_chunks(NUM_RUNS, IMAGES_PER_RUN, A100_BATCH))

print(f"Ready: {len(chunks)} chunks × {len(chunks[0])} batches each")

## Step 7 — Phase 1: AutoProfiler Calibration

Hooks into every Q/K/V projection across all 10 blocks × 6 heads and measures:
- Average Q/K/V spike sparsity
- Per-timestep sparsity evolution
- First-spike times (STBP Q-K temporal gap)
- Dead neuron % (never fires across 500 images)
- Ghost neuron % (fires on zero-input)
- Cross-head Pearson correlation

Output: `gating_profile.json` with a hardware gating policy per head.

In [ ]:
import os, time
from spikegate import AutoProfiler

os.makedirs(REPORT_DIR, exist_ok=True)
PROFILE_JSON = os.path.join(REPORT_DIR, 'gating_profile.json')

print('=' * 60)
print('  Phase 1: AutoProfiler Calibration')
print('=' * 60)

t_cal = time.time()
profiler = AutoProfiler(model, T=T_STEPS)
profile  = profiler.calibrate(
    chunks[0], device=device,
    num_blocks=NUM_BLOCKS, num_heads=NUM_HEADS,
    out_file=PROFILE_JSON,
)
print(f"\nCalibration done in {(time.time() - t_cal) / 60:.1f} min")
print(f"Profile saved  : {PROFILE_JSON}")

# Quick summary of gating policies assigned
from collections import Counter
policies = [
    profile[f'block_{b}'][f'head_{h}']['HARDWARE_GATING_POLICY']
    for b in range(NUM_BLOCKS) for h in range(NUM_HEADS)
]
print("\nGating policy distribution:")
for pol, cnt in Counter(policies).most_common():
    print(f"  {pol:<50s} {cnt:3d} heads")

## Step 8 — Phases 2–5: Full Pipeline

Runs four studies across all 10 data chunks, then generates figures and the report:

| Phase | Description |
|---|---|
| 2 — Stability | Fidelity & compute savings per run |
| 3 — Ablation | Per-policy fidelity/savings with error bars |
| 4 — Importance | MASK-ONE-OUT for all 60 heads (B×H) |
| 5 — Report | 8 figures + comprehensive Markdown |

> **Estimated time on A100**: ~25–35 min (vs. ~90 min on T4)

In [ ]:
import time
from spikegate import DynamicGateController, ReportGenerator

t0 = time.time()

gate_ctrl = DynamicGateController(PROFILE_JSON)
reporter  = ReportGenerator(
    model, gate_ctrl, profile,
    num_blocks=NUM_BLOCKS, num_heads=NUM_HEADS,
    output_dir=REPORT_DIR,
)

report_path = reporter.run_full_pipeline(
    chunks, device=device, images_per_run=IMAGES_PER_RUN,
)

elapsed = time.time() - t0
print(f"\n{'=' * 60}")
print(f"  All phases complete in {elapsed / 60:.1f} minutes")
print(f"{'=' * 60}")
print(f"Report  : {report_path}")
print(f"Figures : {REPORT_DIR}/figures/")
print(f"JSONs   : {REPORT_DIR}/")

## Step 9 — Display Comprehensive Report

In [ ]:
from IPython.display import Markdown, display

with open(report_path, 'r', encoding='utf-8') as f:
    display(Markdown(f.read()))

## Step 10 — Display All Figures

In [ ]:
from IPython.display import Image, display
import glob, os

fig_paths = sorted(glob.glob(f'{REPORT_DIR}/figures/*.png'))
print(f"Found {len(fig_paths)} figures\n")

for fig_path in fig_paths:
    name = os.path.basename(fig_path)
    print(f'── {name} ──')
    display(Image(filename=fig_path, width=920))

## Step 11 — Save All Outputs to Google Drive

In [ ]:
import shutil, os

DRIVE_SAVE = '/content/drive/MyDrive/SpikeGate_MaxFormer_A100/'
os.makedirs(DRIVE_SAVE, exist_ok=True)

shutil.copytree(REPORT_DIR, DRIVE_SAVE, dirs_exist_ok=True)

saved_files = []
for root, _, files in os.walk(DRIVE_SAVE):
    for f in files:
        saved_files.append(os.path.relpath(os.path.join(root, f), DRIVE_SAVE))

print(f"Saved {len(saved_files)} files to Google Drive:")
print(f"  {DRIVE_SAVE}")
print()
for f in sorted(saved_files):
    print(f"  {f}")

---
## Optional — Fast Mode (2 runs × 50 images)

Run this cell instead of Steps 6–8 for a quick smoke test (~2 min on A100).

In [ ]:
# Fast smoke test — overrides NUM_RUNS and IMAGES_PER_RUN for this cell only
import os, time
from torchvision import transforms
from datasets import load_dataset
from spikegate import AutoProfiler, DynamicGateController, ReportGenerator

_FAST_RUNS   = 2
_FAST_IMAGES = 50
_FAST_BATCH  = A100_BATCH
_FAST_DIR    = 'analysis_outputs/fast_test'
os.makedirs(_FAST_DIR, exist_ok=True)

print(f"Fast mode: {_FAST_RUNS} runs × {_FAST_IMAGES} images (batch={_FAST_BATCH})")

fast_chunks = list(create_data_chunks(_FAST_RUNS, _FAST_IMAGES, _FAST_BATCH))

t0 = time.time()
fast_profiler = AutoProfiler(model, T=T_STEPS)
fast_profile  = fast_profiler.calibrate(
    fast_chunks[0], device=device,
    num_blocks=NUM_BLOCKS, num_heads=NUM_HEADS,
    out_file=os.path.join(_FAST_DIR, 'gating_profile.json'),
)

fast_gate = DynamicGateController(os.path.join(_FAST_DIR, 'gating_profile.json'))
fast_rep  = ReportGenerator(
    model, fast_gate, fast_profile,
    num_blocks=NUM_BLOCKS, num_heads=NUM_HEADS,
    output_dir=_FAST_DIR,
)
fast_path = fast_rep.run_full_pipeline(
    fast_chunks, device=device, images_per_run=_FAST_IMAGES,
)
print(f"\nFast run done in {(time.time() - t0) / 60:.1f} min → {fast_path}")

from IPython.display import Markdown, display
with open(fast_path, 'r') as f:
    display(Markdown(f.read()))